In [ ]:
import torch
import torch.nn as nn
import nibabel as nib
import numpy as np

# --- 1. Data Ingestion (NiBabel) ---
def load_fmri_data(file_path):
    img = nib.load(file_path)
    data = img.get_fdata()
    # Normalize data for the VAE
    data = (data - np.min(data)) / (np.max(data) - np.min(data))
    return torch.tensor(data, dtype=torch.float32)

# --- 2. VAE Architecture ---
class BrainVAE(nn.Module):
    def __init__(self):
        super(BrainVAE, self).__init__()
        # Encoder: Downsampling the brain slices
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(16, 32, 3, stride=2, padding=1),
            nn.ReLU()
        )
        # Latent Space (Mean & Log-Variance)
        self.fc_mu = nn.Linear(32 * 23 * 23, 128)
        self.fc_logvar = nn.Linear(32 * 23 * 23, 128)
        
        # Decoder: Reconstruction
        self.decoder_input = nn.Linear(128, 32 * 23 * 23)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(32, 16, 3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(16, 1, 3, stride=2, padding=1, output_padding=1),
            nn.Sigmoid()
        )

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        x = self.encoder(x)
        x = torch.flatten(x, start_dim=1)
        mu, logvar = self.fc_mu(x), self.fc_logvar(x)
        z = self.reparameterize(mu, logvar)
        x = self.decoder_input(z)
        x = x.view(-1, 32, 23, 23)
        return self.decoder(x), mu, logvar

# --- 3. Loss Function (Heavy Reconstruction Focus) ---
def vae_loss(recon_x, x, mu, logvar):
    BCE = nn.functional.binary_cross_entropy(recon_x, x, reduction='sum')
    # KL Weight set to 0.0001 per your project specs
    KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return BCE + (0.0001 * KLD)